# 20. Probability Calibration（spec §15）

## 学習目標
- モデルの確信度が信頼できるか（confidence 0.7 の予測は約70%正しいか）を測る
- ECE（equal-width / equal-mass）・Brier・reliability diagram を読む
- Temperature Scaling を**calibration split で最適化し test split で評価**する（データ分離）

## 数式
次トークンの確信度 $c_t=\max_v p(v\mid x_{<t})$、正誤 $\mathbb{1}[\hat y_t=y_t]$。
ECE $=\sum_b \frac{n_b}{N}\,|\text{acc}_b-\text{conf}_b|$。Temperature scaling: $p_T=\text{softmax}(z/T)$。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
L_RUN = ROOT / "experiments/runs/m4_model_l_modern_seed42"
M5 = ROOT / "experiments/analysis_m5"

In [2]:
cal = load_json(M5 / "calibration.json")
print(f"test top-1 confidence(平均) {cal['test_top1_conf_mean']:.3f} vs 実精度 {cal['test_accuracy']:.3f}")
print(f"raw: NLL {cal['raw']['nll']:.3f}  ECE(width) {cal['raw']['ece_equal_width']:.4f}  Brier {cal['raw']['brier']:.4f}")
print(f"fitted T = {cal['fitted_T']:.3f}（calibration splitで最適化, test splitで評価）")
print(f"scaled: NLL {cal['temperature_scaled']['nll']:.3f}  ECE(mass) {cal['temperature_scaled']['ece_equal_mass']:.4f}")
print(f"accuracy raw {cal['raw']['accuracy']:.4f} vs scaled {cal['temperature_scaled']['accuracy']:.4f}（不変のはず）")

test top-1 confidence(平均) 0.379 vs 実精度 0.380
raw: NLL 3.287  ECE(width) 0.0042  Brier 0.1373
fitted T = 0.984（calibration splitで最適化, test splitで評価）
scaled: NLL 3.287  ECE(mass) 0.0082
accuracy raw 0.3851 vs scaled 0.3851（不変のはず）


In [3]:
# reliability diagram（equal-width）
bins = [b for b in cal["reliability_equal_width"] if b["n"] > 0]
fig = go.Figure()
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode="lines", line=dict(dash="dash", color="#999"), name="完全較正"))
fig.add_trace(go.Scatter(x=[b["avg_conf"] for b in bins], y=[b["acc"] for b in bins],
                         mode="lines+markers", name="実測", line=dict(color="#1f77b4"),
                         marker=dict(size=[max(4, b["n"]**0.5/8) for b in bins])))
fig.update_layout(title="Reliability diagram（対角線=完全較正, 点サイズ=サンプル数）",
                  xaxis_title="平均確信度", yaxis_title="実精度", template="plotly_white", height=420)
fig.show()

In [4]:
# raw vs temperature-scaled の指標比較
raw, sc = cal["raw"], cal["temperature_scaled"]
df = pd.DataFrame({
    "metric": ["NLL","ECE(equal-mass)","accuracy","entropy"],
    "raw": [raw["nll"], sc["ece_equal_mass"] if False else cal["raw"].get("ece_equal_width"), raw["accuracy"], raw["entropy"]],
    "temperature_scaled": [sc["nll"], sc["ece_equal_mass"], sc["accuracy"], sc["entropy"]],
})
df.round(4)

,metric,raw,temperature_scaled
0,NLL,3.2875,3.2865
1,ECE(equal-mass),0.0042,0.0082
2,accuracy,0.3851,0.3851
3,entropy,3.3824,3.3039


## Observation / Interpretation / Caveat
- **Observation**: fitted T と ECE の値は上のセルが実測。Temperature scaling で NLL/ECE が変化し、**accuracy は不変**（argmax を変えないため）。
- **Interpretation**: T>1 なら「過信」、T<1 なら「過小」だった。温度スケーリングは分布の鋭さだけを調整する後処理で、順位や top-1 正解率は変えない。
- **Caveat（重要）**: 次トークン確率の較正は、**回答全体の事実的正しさや hallucination 率とは別物**。よく較正されたモデルでも「もっともらしい誤り」を出す（Model L の生成が示す通り）。T は calibration split で最適化し test で評価した（test で最適化してはいけない）。

**次へ**: [21_generation_anatomy](21_generation_anatomy.ipynb)